# Mem0 with Google Gemini

This cookbook demonstrates how to use **Mem0** as a persistent memory layer for AI conversations powered by **Google Gemini**.

## What you'll learn
- How to configure Mem0 to use Google Gemini as the LLM backend
- How to add memories from conversations
- How to search and retrieve relevant memories
- How to build a full memory-enabled chatbot using Gemini

## Prerequisites
- A [Google AI Studio](https://aistudio.google.com/) API key (free)
- Python 3.9+

## Overview

Mem0 supports Google Gemini as an LLM and embedder provider. This means you can build fully personalized AI assistants without needing an OpenAI key — just your Gemini API key.

```
User Message
     │
     ▼
┌─────────────┐     ┌──────────────┐
│  Mem0 Search│────▶│ Relevant     │
│  (Gemini    │     │ Memories     │
│  Embedder)  │     │              │
└─────────────┘     └──────┬───────┘
                           │
                           ▼
                   ┌───────────────┐
                   │  Gemini LLM   │
                   │  (with memory │
                   │   context)    │
                   └──────┬────────┘
                          │
                          ▼
                   ┌───────────────┐
                   │  Mem0 Add     │
                   │  (store new   │
                   │   memories)   │
                   └───────────────┘
```

## Step 1: Install Dependencies

In [ ]:
!pip install mem0ai google-generativeai

## Step 2: Set Up API Key and Configure Mem0 with Gemini

Get your free Gemini API key from [Google AI Studio](https://aistudio.google.com/app/apikey).

We configure Mem0 to use:
- **LLM**: `gemini-2.0-flash` — fast and capable for chat
- **Embedder**: `models/text-embedding-004` — Google's latest embedding model
- **Vector Store**: In-memory (no extra setup needed for this demo)

In [ ]:
import os
from mem0 import Memory

# Set your Gemini API key
os.environ["GEMINI_API_KEY"] = "your-gemini-api-key-here"  # Replace with your key

# Configure Mem0 to use Gemini for both LLM and embeddings
config = {
    "llm": {
        "provider": "gemini",
        "config": {
            "model": "gemini-2.0-flash",
            "api_key": os.environ["GEMINI_API_KEY"],
            "temperature": 0.1,
            "max_tokens": 2000,
        },
    },
    "embedder": {
        "provider": "gemini",
        "config": {
            "model": "models/text-embedding-004",
            "api_key": os.environ["GEMINI_API_KEY"],
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "gemini_memories",
            "host": "localhost",
            "port": 6333,
            "embedding_model_dims": 768,  # Gemini text-embedding-004 dimension
        },
    },
}

# Initialize memory
memory = Memory.from_config(config)
print("✅ Mem0 initialized with Google Gemini!")

> **Note:** The above uses a local Qdrant instance. If you don't have Qdrant running, you can use the in-memory vector store instead (great for quick testing):
>
> ```python
> "vector_store": {
>     "provider": "memory",
>     "config": {"embedding_model_dims": 768}
> }
> ```
> Start Qdrant with Docker: `docker run -p 6333:6333 qdrant/qdrant`

## Step 3: Adding Memories

Mem0 automatically extracts and stores facts from conversations. You can add memories from plain text or structured chat messages.

In [ ]:
# Define a user ID — memories are scoped per user
USER_ID = "alice"

# Add memories from a conversation
conversation = [
    {"role": "user", "content": "Hi! My name is Alice and I'm a software engineer."},
    {"role": "assistant", "content": "Nice to meet you, Alice! What kind of software do you work on?"},
    {"role": "user", "content": "I work on backend systems using Python and FastAPI. I love clean code and hate spaghetti code."},
    {"role": "assistant", "content": "That's great! Python and FastAPI are a powerful combination for backend development."},
    {"role": "user", "content": "I also enjoy hiking on weekends and I'm vegetarian."},
]

result = memory.add(conversation, user_id=USER_ID)
print("Memories added:")
for mem in result.get("results", []):
    print(f"  - [{mem['event']}] {mem['memory']}")

In [ ]:
# You can also add a single text memory directly
result2 = memory.add(
    "Alice prefers dark mode in all her development tools and uses VS Code as her primary editor.",
    user_id=USER_ID
)
print("Additional memory added:")
for mem in result2.get("results", []):
    print(f"  - [{mem['event']}] {mem['memory']}")

In [ ]:
# View all stored memories for the user
all_memories = memory.get_all(user_id=USER_ID)
print(f"\nAll memories for user '{USER_ID}':")
for i, mem in enumerate(all_memories.get("results", []), 1):
    print(f"  {i}. {mem['memory']}")

## Step 4: Searching Memories

Mem0 uses semantic search powered by Gemini embeddings to find the most relevant memories for any query.

In [ ]:
# Search for memories relevant to a specific topic
queries = [
    "What does Alice do for work?",
    "What are Alice's hobbies?",
    "What tools does Alice use for coding?",
    "What are Alice's food preferences?",
]

for query in queries:
    results = memory.search(query=query, user_id=USER_ID, limit=2)
    print(f"\n🔍 Query: '{query}'")
    for mem in results.get("results", []):
        score = mem.get('score', 'N/A')
        print(f"   → {mem['memory']} (score: {score:.3f})")

## Step 5: Full Memory-Enabled Chatbot with Gemini

Now let's put it all together — a chatbot that:
1. Retrieves relevant memories before every response
2. Uses Gemini to generate a personalized answer
3. Automatically stores new information from the conversation

In [ ]:
import google.generativeai as genai

# Initialize Gemini client
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
gemini_model = genai.GenerativeModel("gemini-2.0-flash")


def chat_with_memory(message: str, user_id: str = USER_ID) -> str:
    """
    Send a message to Gemini with Mem0-powered persistent memory.
    
    Args:
        message: The user's message
        user_id: Unique identifier for the user
    
    Returns:
        The assistant's response string
    """
    # Step 1: Retrieve relevant memories for context
    relevant_memories = memory.search(query=message, user_id=user_id, limit=5)
    memories_text = "\n".join(
        f"- {m['memory']}" for m in relevant_memories.get("results", [])
    )

    # Step 2: Build prompt with memory context
    system_prompt = """You are a helpful, personalized AI assistant. 
You have access to the user's memory and should use it to provide personalized responses.
Always be warm, concise, and relevant. Reference memories naturally when appropriate."""

    if memories_text:
        full_prompt = f"""{system_prompt}

What you remember about this user:
{memories_text}

User: {message}
Assistant:"""
    else:
        full_prompt = f"""{system_prompt}

User: {message}
Assistant:"""

    # Step 3: Generate response using Gemini
    response = gemini_model.generate_content(full_prompt)
    assistant_reply = response.text

    # Step 4: Store the new conversation turn in memory
    new_turn = [
        {"role": "user", "content": message},
        {"role": "assistant", "content": assistant_reply},
    ]
    memory.add(new_turn, user_id=user_id)

    return assistant_reply


print("✅ Chatbot ready! Memory-enabled conversations with Gemini.")

In [ ]:
# Test the chatbot — it will personalize responses using Alice's stored memories

test_messages = [
    "Can you recommend a good lunch option for me?",
    "What IDE settings would you suggest for my workflow?",
    "I'm thinking about learning a new framework. Any suggestions based on my background?",
]

for msg in test_messages:
    print(f"\n👤 User: {msg}")
    reply = chat_with_memory(msg)
    print(f"🤖 Gemini: {reply}")
    print("-" * 60)

In [ ]:
# Interactive chat loop — uncomment and run for a live session

# print("Starting interactive chat (type 'quit' to exit)")
# print("=" * 50)
# while True:
#     user_input = input("You: ").strip()
#     if not user_input:
#         continue
#     if user_input.lower() in ("quit", "exit", "bye"):
#         print("Goodbye! Your memories are saved for next time.")
#         break
#     response = chat_with_memory(user_input)
#     print(f"Gemini: {response}\n")

## Summary

In this cookbook you learned how to:

| Step | What we did |
|------|-------------|
| Configure | Set up Mem0 with Gemini LLM + Gemini Embedder |
| Add memories | Store facts from conversations and plain text |
| Search memories | Semantically retrieve relevant context |
| Build chatbot | Create a full personalized assistant with persistent memory |

## Next Steps

- **Persist to a database**: Swap the in-memory vector store for [Qdrant](https://qdrant.tech/) or [Supabase](https://supabase.com/) for production use
- **Multi-user support**: Use different `user_id` values to maintain separate memories per user
- **Graph memory**: Enable Mem0's graph memory feature for relationship-aware recall
- **Explore more integrations**: Check out the [Mem0 docs](https://docs.mem0.ai) for LangChain, CrewAI, and AutoGen integrations

## Resources

- [Mem0 Documentation](https://docs.mem0.ai)
- [Google Gemini API](https://ai.google.dev/)
- [Google AI Studio (get API key)](https://aistudio.google.com/app/apikey)
- [Mem0 GitHub](https://github.com/mem0ai/mem0)